In [ ]:
# Cell 1: Install the PEFT Stack
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q -U trl
!pip install -q -U datasets
!pip install -q -U accelerate

print("✅ PEFT Stack successfully installed!")

In [ ]:
# Cell 2: Authenticate and Load Data
from huggingface_hub import login
from datasets import load_dataset
import os

# Put your Hugging Face Write Token here
hf_token = "Your_hugging_face_token"
login(token=hf_token)

# Replace with the actual Repo ID you used in Week 8
# e.g., "your-username/pii-redactor-training-v1"
REPO_ID = "your-hugging-face-username/pii-redactor-training-v1"

print(f"Pulling {REPO_ID} from the Hub...")
# Load the dataset directly into the Colab GPU memory
dataset = load_dataset(REPO_ID, split="train")

print(f"✅ Successfully loaded {len(dataset)} training rows.")
print("\nSample Row:")
print(dataset[0]['messages'])

In [ ]:
# Cell 3: Load the Tokenizer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# We will use Mistral 7B Instruct v0.3
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_auth_token=hf_token)

# Setting up padding (Critical for batch training)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fixes weird formatting bugs during training

print("✅ Tokenizer loaded successfully!")

In [ ]:
# Cell 4: Define the 4-bit Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True, # Squeezes out even more memory!
    bnb_4bit_quant_type="nf4",      # NormalFloat4 - optimized for neural net weights
    bnb_4bit_compute_dtype=torch.bfloat16 # The math is still done in 16-bit for accuracy
)
print("✅ BitsAndBytes Configuration set!")

In [ ]:
# Cell 5: Load the Base Model
print("Downloading Base Model into 4-bit memory... (This takes a few minutes)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto", # Automatically puts the model on the GPU
    token=hf_token
)

# Disable caching to save VRAM during training
model.config.use_cache = False

print("✅ Model successfully loaded into 4-bit VRAM!")

In [ ]:
# Cell 6: Prepare for k-bit Training
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# Enable gradient checkpointing to save massive amounts of VRAM
model.gradient_checkpointing_enable()

# Prep the model for quantized training
model = prepare_model_for_kbit_training(model)
print("✅ Model prepped for gradient calculations.")

In [ ]:
# Cell 7: Define and Attach the LoRA Adapters
peft_config = LoraConfig(
    r=16,                       # The rank of the update matrices
    lora_alpha=32,              # The scaling factor (usually 2x the rank)
    target_modules=[            # Target all linear layers for maximum intelligence
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,          # Drop 5% of neurons randomly to prevent overfitting
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach the adapters to the 4-bit base model
model = get_peft_model(model, peft_config)

# Let's see exactly how many parameters we are actually training
model.print_trainable_parameters()

In [ ]:
# Cell 8: Dataset Formatting
def format_chat_template(row):
    chat = row['messages']
    formatted_prompt = tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": formatted_prompt}

print("Formatting dataset to match Llama-3's prompt structure...")
formatted_dataset = dataset.map(format_chat_template)
print("✅ Formatting complete!")

In [ ]:
# Cell 9: Define the Training Arguments
from trl import SFTConfig, SFTTrainer

training_arguments = SFTConfig(
    output_dir="./pii_redactor_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=5,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    lr_scheduler_type="constant",
    train_sampling_strategy="group_by_length",
    dataset_text_field="text",
    max_length=1024
)

print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    args=training_arguments,
)
print("✅ Trainer ready.")

In [ ]:
# Cell 10: TRAIN
print("🚀 Commencing Neural Surgery (Training Started)...")
trainer.train()

In [ ]:
# Cell 11: Save the Proprietary Brain
ADAPTER_NAME = "pii-redactor-mistral-lora-v1"

print(f"Saving the LoRA adapters to {ADAPTER_NAME}...")
trainer.model.save_pretrained(ADAPTER_NAME)
tokenizer.save_pretrained(ADAPTER_NAME)

print("✅ Adapters successfully extracted and saved to disk!")

In [ ]:
# Cell 12: The Batch Evaluation Matrix
from datasets import load_dataset
import torch
from tqdm.notebook import tqdm
import re

DATASET_ID = "Your-Username/pii-redactor-training-v1"
print(f"Downloading Holdout Test Set from {DATASET_ID}...")
test_dataset = load_dataset(DATASET_ID, split="test")
print(f"Loaded {len(test_dataset)} test examples.\n")

exact_matches = 0
total_examples = len(test_dataset)
leak_failures = 0

print("Commencing Batch Inference on Test Set...")

for i, row in enumerate(tqdm(test_dataset, desc="Evaluating")):
    messages = row['messages']

    # Extract Ground Truth
    system_prompt = messages[0]['content']
    raw_text = messages[1]['content']
    ground_truth = messages[2]['content'].strip()

    # Format for Mistral inference
    eval_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": raw_text}
    ]
    prompt = tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate Prediction
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=250,
            temperature=0.1,             # Prevent deterministic loops
            repetition_penalty=1.1,      # Break asterisk repetition
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    prediction = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

    # --- METRIC 1: Exact Match ---
    if prediction == ground_truth:
        exact_matches += 1

    # --- METRIC 2: Tag Recall (Leak Detection) ---
    tags_pattern = r'\[(?:NAME|SSN|ADDRESS|PHONE|EMAIL|DOB|MRN)\]'
    gt_tags = re.findall(tags_pattern, ground_truth)
    pred_tags = re.findall(tags_pattern, prediction)

    if len(pred_tags) < len(gt_tags):
        leak_failures += 1

# Calculate Final Metrics
exact_match_accuracy = (exact_matches / total_examples) * 100
leak_rate = (leak_failures / total_examples) * 100
safe_rate = 100 - leak_rate

print("\n" + "="*40)
print("🏆 RE-EVALUATION METRICS 🏆")
print("="*40)
print(f"Total Test Examples: {total_examples}")
print(f"Strict Exact Match:  {exact_match_accuracy:.2f}%")
print(f"Data Safety Rate:    {safe_rate:.2f}% (Rows with zero tag leaks)")
print(f"Critical Leak Rate:  {leak_rate:.2f}% (Rows where PII was potentially missed)")
print("="*40)

In [ ]:
# Cell 13: The Failure Inspector
print("🔍 INSPECTING THE LEAKS 🔍\n")

failures_logged = 0
max_failures_to_print = 5 # We only want to look at a few to diagnose the issue

for i, row in enumerate(test_dataset):
    if failures_logged >= max_failures_to_print:
        break

    messages = row['messages']
    raw_text = messages[1]['content']
    ground_truth = messages[2]['content'].strip()

    # Format and Generate (Using the safe parameters)
    eval_messages = [{"role": "system", "content": messages[0]['content']}, {"role": "user", "content": raw_text}]
    prompt = tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.1,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    prediction = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

    # Tag Counting Logic
    tags_pattern = r'\[(?:NAME|SSN|ADDRESS|PHONE|EMAIL|DOB|MRN)\]'
    gt_tags = re.findall(tags_pattern, ground_truth)
    pred_tags = re.findall(tags_pattern, prediction)

    # If the model missed a tag, PRINT THE EVIDENCE
    if len(pred_tags) < len(gt_tags):
        print(f"--- FAILURE CAUGHT (Row {i}) ---")
        print(f"Missing Tags: Ground Truth expected {len(gt_tags)}, Model provided {len(pred_tags)}")
        print("\nGROUND TRUTH EXPECTED:")
        print(ground_truth[:300] + "...") # Print first 300 chars to save screen space
        print("\nMODEL PREDICTION:")
        print(prediction[:300] + "...")
        print("="*50 + "\n")
        failures_logged += 1

In [ ]:
# Cell 14: Push the Adapters to the Hub
# Replace 'your-username' with your actual Hugging Face username
REPO_ID = "your-hugging-face-username/pii-redactor-mistral-lora-v1"

print(f"Pushing adapters to Hugging Face Hub at {REPO_ID}...")

# Push the model weights
trainer.model.push_to_hub(REPO_ID, private=True)

# Push the tokenizer
tokenizer.push_to_hub(REPO_ID, private=True)

print("✅ SUCCESS! Your proprietary AI is now safely stored in the cloud.")